# Phase 3 - RAGAS Evaluation (AmazonQA-RAG)

**Goal:** Quantitatively compare three systems on a held-out Amazon QA test split:

1. **`base`** - Base instruction-tuned LLM, no retrieval, no fine-tuning
2. **`rag_base`** - RAG (FAISS + base LLM), no fine-tuning
3. **`rag_finetuned`** - RAG + our QLoRA-fine-tuned LLM (the hero system)

**Metrics (RAGAS):**

| Metric | What it measures |
|---|---|
| `faithfulness` | Does the generated answer **stay grounded** in the retrieved context? (Anti-hallucination.) |
| `answer_relevancy` | Does the answer **actually address** the question? |
| `context_precision` | Are the **top-ranked retrieved chunks** the relevant ones? |
| `context_recall` | Did retrieval surface **all** the information needed for the gold answer? |

Each metric is in `[0, 1]`; higher is better.

---

**Prerequisites:** Phase 1 adapter at `artifacts/lora_adapter/` and Phase 2 FAISS index at `artifacts/faiss_index/`. If either is missing, the notebook will rebuild the FAISS index automatically; you must run notebook `01_qlora_finetune.ipynb` first if the adapter is absent.

**Runtime:** ~15-25 min for `n_samples=100` on a Colab T4 (dominated by RAGAS LLM-as-judge calls).

## Step 1 - Install RAGAS + matplotlib

RAGAS pulls in its own dependency tree; we pin a known-good version that plays nicely with the LangChain 0.2.x stack from Phase 2.

In [ ]:
%pip install -q ragas==0.1.21 matplotlib==3.9.2 datasets==2.21.0
print('RAGAS install complete')

## Step 2 - Mount project root

Adjust `PROJECT_ROOT` if you cloned the repo to a different location.

In [ ]:
import os, sys
from pathlib import Path

# Auto-detect: Colab vs local
if Path('/content').exists():
    PROJECT_ROOT = Path('/content/AmazonQA-RAG')
    if not PROJECT_ROOT.exists():
        # Convenience: mount Drive if you backed the repo up there
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
        # Replace with your own path:
        # !cp -r '/content/drive/MyDrive/AmazonQA-RAG' /content/
else:
    PROJECT_ROOT = Path.cwd()
    while not (PROJECT_ROOT / 'configs' / 'config.py').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)
assert (PROJECT_ROOT / 'evaluation').exists(), 'evaluation/ package not found - check the path'

## Step 3 - (Optional) Configure RAGAS judge backend

RAGAS uses an LLM-as-judge for `faithfulness` + `answer_relevancy`.

- **Recommended:** set an `OPENAI_API_KEY` Colab secret -> RAGAS uses `gpt-4o-mini` (~$0.10 for n=100).
- **Free fallback:** leave it unset -> RAGAS uses a local `flan-t5-base` judge (slower, weaker, but $0).

The default `cfg.eval.ragas_judge_backend = 'auto'` picks the right one automatically.

In [ ]:
import os

# Colab: read the OPENAI_API_KEY secret if present
try:
    from google.colab import userdata  # type: ignore
    key = userdata.get('OPENAI_API_KEY')
    if key:
        os.environ['OPENAI_API_KEY'] = key
        print('OpenAI key loaded from Colab secret -> RAGAS will use gpt-4o-mini')
    else:
        print('No OPENAI_API_KEY secret -> RAGAS will fall back to local flan-t5 judge')
except ImportError:
    if 'OPENAI_API_KEY' in os.environ:
        print('OpenAI key already set in env')
    else:
        print('No OpenAI key -> using local flan-t5 judge')

## Step 4 - Load configuration

Set `USE_PHI2 = True` if you fine-tuned with the Phi-2 fallback in Phase 1.

In [ ]:
from configs.config import get_default_config, get_phi2_fallback_config

USE_PHI2 = False  # flip True if you trained Phi-2 in Phase 1

cfg = get_phi2_fallback_config() if USE_PHI2 else get_default_config()
cfg.paths.ensure_all()

print('Base model :', cfg.model.base_model_id)
print('Adapter dir:', cfg.paths.adapter_dir, '(exists:', cfg.paths.adapter_dir.exists(), ')')
print('FAISS dir  :', cfg.paths.vector_store_dir, '(exists:', (cfg.paths.vector_store_dir / 'index.faiss').exists(), ')')
print('Test set N :', cfg.eval.test_set_size)
print('Metrics    :', cfg.eval.metrics)

## Step 5 - Build the held-out test set

We pull a fresh slice of `embedding-data/Amazon-QA` using a **different seed** from the one Phase 1 used, AND filter out any question text that appeared in the training set. The resulting CSV is reusable across runs - only rebuild it if you want a different `n_samples` or seed.

In [ ]:
from evaluation.test_set import build_test_set, load_test_set

# Smaller sample for the notebook walkthrough; the CLI defaults to 100.
TEST_N = 50

test_df = build_test_set(cfg, n_samples=TEST_N, overwrite=False)
print(f'Test set rows: {len(test_df)}')
test_df.head(3)

## Step 6 - Spot-check the three systems on one question

Sanity check that all three pipelines load and answer ONE question before we kick off the full RAGAS run. If any of these errors out, fix it before spending a 20-minute eval budget.

In [ ]:
from evaluation.generators import GeneratorRegistry, SYSTEM_NAMES

registry = GeneratorRegistry(cfg)
sample_q = test_df.iloc[0]['question']
print('Q:', sample_q)
print('-' * 60)

for sys_name in SYSTEM_NAMES:
    out = registry.generate(sys_name, sample_q)
    print(f'\n[{sys_name}]')
    print(out['answer'][:500])
    print(f'(retrieved {len(out["contexts"])} chunks)')
    registry.unload(sys_name)  # free VRAM between systems

## Step 7 - Run RAGAS on one system (debugging shortcut)

Useful if you suspect a problem in a single pipeline. Skip to Step 8 for the full run.

In [ ]:
# Toggle to True to run this debug step
RUN_SINGLE_SYSTEM_DEBUG = False

if RUN_SINGLE_SYSTEM_DEBUG:
    from evaluation.ragas_eval import evaluate_system_with_ragas

    debug_scores = evaluate_system_with_ragas(
        system_name='rag_finetuned',
        qa_df=test_df,
        n_samples=5,             # tiny slice for debugging
        cfg=cfg,
        registry=registry,
    )
    debug_scores.tail()

## Step 8 - Full three-way RAGAS evaluation

This is the headline run. Outputs:

- `results/base_ragas_scores.csv`
- `results/rag_base_ragas_scores.csv`
- `results/rag_finetuned_ragas_scores.csv`
- `results/ragas_summary.csv` (one row per system, columns = `<metric>_mean`)
- `results/ragas_comparison.png` (grouped bar chart)
- `results/ragas_radar.png` (radar chart)

**Time budget:** ~15-25 min for n=100, scaling roughly linearly.

In [ ]:
from evaluation.ragas_eval import run_full_evaluation

summary = run_full_evaluation(
    cfg,
    n_samples=TEST_N,        # use TEST_N here for a faster demo; bump to 100+ for the report
    smoke_test=False,
    save_plots=True,
)
summary.round(3)

## Step 9 - Inspect the saved plots inline

In [ ]:
from IPython.display import Image, display

display(Image(str(cfg.paths.results_dir / cfg.eval.bar_chart_filename)))
display(Image(str(cfg.paths.results_dir / cfg.eval.radar_chart_filename)))

## Step 10 - (Colab) Backup results to Drive

Persist the CSVs + PNGs to Drive so you don't lose them when the runtime resets.

In [ ]:
import shutil
from pathlib import Path

if Path('/content').exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=False)
        target = Path('/content/drive/MyDrive/AmazonQA-RAG-Phase3-Results')
        target.mkdir(parents=True, exist_ok=True)
        for p in cfg.paths.results_dir.iterdir():
            shutil.copy2(p, target / p.name)
        print('Backed up to', target)
    except Exception as e:
        print('Drive backup skipped:', e)
else:
    print('Local run - results live in', cfg.paths.results_dir)

## Troubleshooting

| Symptom | Fix |
|---|---|
| `ImportError: ragas` | Re-run Step 1's pip cell. |
| `OPENAI_API_KEY error` even though set | Colab caches `userdata` between sessions; restart runtime, re-run Step 3. |
| `OOM` during generation | The registry evicts each system before loading the next - if it still OOMs, set `cfg.eval.eval_max_new_tokens=128` and re-run. |
| `RAGAS metric returned NaN` | Usually a malformed context. Check the per-system CSV for the offending row; RAGAS should also log a warning since `raise_exceptions=False`. |
| `FAISS index not found` | Run `notebooks/02_build_rag.ipynb` first, OR delete `artifacts/faiss_index/` and let Phase 3 rebuild it. |
| `Adapter not found` for `rag_finetuned` | Run `notebooks/01_qlora_finetune.ipynb` first. As a workaround, evaluate only `base` + `rag_base` by editing the call to `run_full_evaluation(systems=['base', 'rag_base'])`. |

## Reproducibility

- Test-set seed: `cfg.eval.test_set_seed = 2026` (different from training seed).
- Generation `temperature = 0.3` (low but not greedy - keeps RAGAS scores stable but allows some variation).
- All metric means are saved to `results/ragas_summary.csv` for paste-ready table generation.
- See the `RAGAS Evaluation` section of the README for the metric definitions and result table.